# Ray Data + TorchRec + Ray Train Learning

This notebook demonstrates distributed training using:
- **Ray Data** for efficient parquet file reading and preprocessing
- **TorchRec** for optimized embedding operations with `EmbeddingBagCollection`
- **Ray Train** for multi-worker distributed training

**Key features**: Distributed data loading, sharded embeddings, multi-worker training, and throughput benchmarking.

**This notebook needs to be run in the Google Colab**

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Setup working directory in Google Drive
import os
assert os.path.exists('/content/drive')
WORK_DIR = '/content/drive/MyDrive/colab/torch_data_pipe_learning'
%mkdir -p $WORK_DIR

In [ ]:
import shutil
import os

RAY_LEARNING_DIR = WORK_DIR + '/ray_learning'

# Remove existing folder if it exists
if os.path.exists(RAY_LEARNING_DIR):
    print(f"Removing existing {RAY_LEARNING_DIR} folder...")
    shutil.rmtree(RAY_LEARNING_DIR)

# Create fresh folder
os.makedirs(RAY_LEARNING_DIR)
print(f"✅ Created fresh {RAY_LEARNING_DIR} folder")

In [ ]:
# 5. Setup src imports
print("\n[5] Setting up src imports...")
# Clone repo for src code
repo_url = 'https://github.com/allyoushawn/jupyter_notebook_projects.git'
repo_dir = 'jupyter_notebook_projects'
branch_name = 'yw/colab_ray_train'


if os.path.exists(repo_dir):
    !rm -rf $repo_dir
!git clone -q $repo_url
%cd $repo_dir
!git fetch --all
!git checkout $branch_name

# src_path = os.path.abspath('tiger_semantic_id/src')
# if src_path not in sys.path:
#     sys.path.insert(0, src_path)
# print(f"✅ Added src to path: {src_path}")


In [ ]:
%cd ml_misc/torch_data_pipe_learning/

In [ ]:
!pip install -r requirements_torchrec_colab.txt

In [ ]:
# Generate dummy data for petastorm learning
# This will create parquet files in the data/ directory following the structure:
# data/ds=YYYYMMDD/h=HH/<uuid>.parquet

#!python dummy_data_gen.py --output-dir $RAY_LEARNING_DIR/data --start-date 20260101 --end-date 20260114 --null-probability 0.05

In [ ]:
# Install TorchRec and dependencies
# TorchRec requires fbgemm-gpu which is only available on Linux/CUDA
# On macOS, we install torchrec with --no-deps to bypass the fbgemm-gpu requirement
#
# IMPORTANT: PyTorch 2.9.0 may not have compatible fbgemm-gpu yet. 
# This cell will downgrade to PyTorch 2.5.1 if needed for compatibility.
# After running this cell, you may need to RESTART THE RUNTIME for changes to take effect.

# import subprocess
# import sys

# def is_package_installed(package_name):
#     """Check if a package is installed."""
#     try:
#         __import__(package_name.replace("-", "_"))
#         return True
#     except ImportError:
#         return False

# def is_torchrec_working():
#     """Check if TorchRec can be imported without errors (including OSError from fbgemm)."""
#     try:
#         from torchrec import EmbeddingBagCollection
#         return True
#     except (ImportError, OSError):
#         return False

# # Check if CUDA is available (need torch first)
# import torch
# cuda_available = torch.cuda.is_available()
# torch_version = torch.__version__

# print(f"PyTorch version: {torch_version}")
# print(f"CUDA available: {cuda_available}")

# if cuda_available:
#     print("\nCUDA detected - checking TorchRec compatibility...")
    
#     if is_torchrec_working():
#         print("TorchRec already available and working!")
#     else:
#         print("TorchRec not working - attempting to install compatible versions...")
        
#         # Check if PyTorch version is too new (2.6+ doesn't have stable fbgemm-gpu yet)
#         major, minor = map(int, torch_version.split('.')[:2])
        
#         if major >= 2 and minor >= 6:
#             print(f"\nPyTorch {torch_version} is too new - no compatible fbgemm-gpu available.")
#             print("Downgrading to PyTorch 2.5.1 with matching torchrec/fbgemm-gpu...")
#             print("⚠️  IMPORTANT: After this cell completes, RESTART THE RUNTIME!\n")
            
#             # Uninstall current versions
#             !pip uninstall torch torchvision torchaudio torchrec fbgemm-gpu -y -q 2>/dev/null || true
            
#             # Install PyTorch 2.5.1 + compatible torchrec ecosystem
#             # PyTorch 2.5.1 has stable fbgemm-gpu 1.0.0 support
#             !pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
#             !pip install fbgemm-gpu==1.0.0 --index-url https://download.pytorch.org/whl/cu121 -q
#             !pip install torchrec==1.0.0 --index-url https://download.pytorch.org/whl/cu121 -q
#             !pip install torchmetrics iopath -q
            
#             print("\n" + "="*60)
#             print("Installation complete!")
#             print("⚠️  PLEASE RESTART THE RUNTIME NOW (Runtime -> Restart runtime)")
#             print("Then re-run the cells from the beginning.")
#             print("="*60)
#         else:
#             # For PyTorch 2.5.x or earlier, try direct installation
#             print("Trying to install matching torchrec...")
#             !pip uninstall torchrec fbgemm-gpu -y -q 2>/dev/null || true
#             !pip install torchrec torchmetrics iopath -q
            
#             if is_torchrec_working():
#                 print("TorchRec installed successfully!")
#             else:
#                 print("WARNING: TorchRec installation may have issues.")
# else:
#     # On macOS/CPU, install torchrec without deps to skip fbgemm-gpu
#     print("\nNo CUDA detected (macOS/CPU) - installing torchrec without fbgemm-gpu...")
    
#     # Install torchrec dependencies first (excluding fbgemm-gpu)
#     if not is_package_installed("torchmetrics"):
#         !pip install torchmetrics -q
#     if not is_package_installed("iopath"):
#         !pip install iopath -q
    
#     # Install torchrec with --no-deps to bypass fbgemm-gpu requirement
#     if not is_package_installed("torchrec"):
#         !pip install torchrec --no-deps -q
    
#     print("TorchRec installed successfully (without fbgemm-gpu)")

# Verify installation (may fail if runtime restart is needed)
try:
    from torchrec import EmbeddingBagCollection
    print("\nVerification: TorchRec imported successfully!")
except ImportError as e:
    print(f"\nNote: TorchRec import failed: {e}")
    print("If you just downgraded PyTorch, restart the runtime and re-run.")
except OSError as e:
    print(f"\nNote: TorchRec library loading failed: {e}")
    print("If you just downgraded PyTorch, restart the runtime and re-run.")

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import ray
from ray.data import read_parquet
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, get_dataset_shard
import ray.train as train

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
import time
import hashlib

# TorchRec imports
from torchrec import EmbeddingBagCollection, EmbeddingBagConfig
from torchrec.sparse.jagged_tensor import KeyedJaggedTensor
from torchrec.modules.crossnet import CrossNet as TorchRecCrossNet
from torchrec.sparse.jagged_tensor import KeyedTensor

# Feature config
from feature_config import FEATURE_CONFIGS, FeatureType

In [ ]:
# Initialize Ray (local mode for development)
# Set num_cpus to limit resource usage if needed
ray.init(
    num_cpus=4,
    ignore_reinit_error=True,
    _temp_dir="/tmp/ray"
)

print(f"Ray initialized: {ray.is_initialized()}")
print(f"Ray cluster resources: {ray.cluster_resources()}")

In [ ]:
# Device Auto-Detection: CUDA > MPS > CPU
def get_device():
    """Auto-detect best available device: CUDA > MPS > CPU"""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print("  fbgemm-gpu enabled for optimized CUDA embedding operations")
elif device.type == "mps":
    print(f"  Apple Silicon GPU (Metal Performance Shaders)")
    print("  Note: fbgemm-gpu not available for MPS, using CPU fallback for embeddings")
else:
    print(f"  CPU")

## Section 1: Ray Data Loading

Ray Data provides distributed parquet reading with automatic sharding across workers.

In [ ]:
# Hash function for string bucketization (same as petastorm_learning.ipynb)
def hash_string_to_bucket(s: str, num_buckets: int) -> int:
    """
    Hash string to bucket index 1~num_buckets. Returns 0 for null/empty.
    
    Args:
        s: String to hash (can be None or empty)
        num_buckets: Number of buckets (excluding index 0 reserved for null)
    
    Returns:
        Bucket index: 0 for null/empty, 1 to num_buckets for valid strings
    """
    if s is None or s == "":
        return 0
    hash_val = int(hashlib.md5(s.encode('utf-8')).hexdigest(), 16)
    return (hash_val % num_buckets) + 1  # 1 to num_buckets, 0 reserved for null

def process_var_len_sparse(str_list, max_len: int, num_buckets: int):
    """Process a list of strings: hash, trim to max_len, pad with 0s."""
    if str_list is None:
        str_list = []
    # Hash each string to bucket index
    indices = [hash_string_to_bucket(s, num_buckets) for s in str_list]
    # Trim to max_len (keep first max_len elements)
    indices = indices[:max_len]
    # Pad with 0s if shorter than max_len
    indices += [0] * (max_len - len(indices))
    return np.array(indices, dtype=np.int64)

print("Hash utilities ready")

In [ ]:
# Preprocessing function for Ray Data (equivalent to Petastorm's TransformSpec)
def preprocess_batch(batch: dict) -> dict:
    """
    Ray Data batch transform - runs on each worker.
    
    This function processes batches from Ray Data, handling null values and
    converting features according to FEATURE_CONFIGS.
    
    Handles based on FeatureType:
    - DENSE: Fill NaN with 0.0
    - EMBEDDING: Convert to float32, replace null with zero vector
    - SPARSE: Hash strings to bucket indices (int64)
    - VAR_LEN_SPARSE: Hash strings, trim/pad to max_len (int64 array)
    """
    result = {}
    
    for col, config in FEATURE_CONFIGS.items():
        if col not in batch:
            continue
        
        values = batch[col]
        
        if config.type == FeatureType.DENSE:
            # Handle scalar feature columns (fill NaN with 0.0)
            if isinstance(values, np.ndarray):
                result[col] = np.nan_to_num(values, nan=0.0).astype(np.float32)
            else:
                # Handle list/Series
                result[col] = np.array([float(v) if v is not None else 0.0 for v in values], dtype=np.float32)
        
        elif config.type == FeatureType.EMBEDDING:
            # Handle numeric embedding columns (list<double>)
            # Convert all values to float32 consistently
            emb_dim = config.dim
            if isinstance(values, np.ndarray) and values.ndim == 2:
                # Already a 2D array
                result[col] = np.nan_to_num(values, nan=0.0).astype(np.float32)
            else:
                # List of arrays/lists
                processed = []
                for x in values:
                    if x is None or (isinstance(x, float) and np.isnan(x)):
                        processed.append(np.zeros(emb_dim, dtype=np.float32))
                    else:
                        arr = np.asarray(x, dtype=np.float32)
                        if len(arr) != emb_dim:
                            # Pad or trim to match dimension
                            if len(arr) < emb_dim:
                                arr = np.pad(arr, (0, emb_dim - len(arr)), 'constant')
                            else:
                                arr = arr[:emb_dim]
                        processed.append(arr)
                result[col] = np.stack(processed)
        
        elif config.type == FeatureType.SPARSE:
            # Handle SPARSE columns (single strings -> bucket indices)
            num_buckets = config.num_buckets
            result[col] = np.array([hash_string_to_bucket(x, num_buckets) for x in values], dtype=np.int64)
        
        elif config.type == FeatureType.VAR_LEN_SPARSE:
            # Handle VAR_LEN_SPARSE columns (list of strings -> padded/trimmed bucket indices)
            max_len = config.max_len
            num_buckets = config.num_buckets
            result[col] = np.stack([process_var_len_sparse(x, max_len, num_buckets) for x in values])
        
        else:
            # ID, PARTITION, LABEL - pass through
            if isinstance(values, np.ndarray):
                result[col] = values
            else:
                result[col] = np.array(values)
    
    return result

print("Preprocessing function ready")

In [ ]:
# Load data using Ray Data
data_path = Path(f"{RAY_LEARNING_DIR}/data").resolve()

# Use glob to explicitly list all parquet files (similar to petastorm approach)
all_parquet_files = sorted(data_path.glob("**/*.parquet"))
print(f"Found {len(all_parquet_files)} parquet files via glob")

# Convert to list of paths for Ray Data
file_paths = [str(f.absolute()) for f in all_parquet_files]

# Read parquet files with Ray Data
# Ray Data automatically handles sharding across workers
ds = read_parquet(file_paths)

# Apply preprocessing transformation
ds = ds.map_batches(preprocess_batch, batch_format="numpy")

print(f"Dataset created: {ds}")
print(f"Dataset schema: {ds.schema()}")

In [ ]:
# Test reading a batch from Ray Data
print("Testing Ray Data batch reading...")
batch = ds.take_batch(batch_size=8)

print(f"\nBatch keys: {list(batch.keys())}")
print(f"\nBatch shapes:")
for k, v in batch.items():
    if isinstance(v, np.ndarray):
        print(f"  {k}: shape {v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v)}")

# Show sample data
if 'ds' in batch:
    print(f"\nDate in this batch: {batch['ds'][0] if len(batch['ds']) > 0 else 'N/A'}")
if 'emb_1' in batch:
    print(f"\nEmbedding features:")
    print(f"  emb_1: shape {batch['emb_1'].shape}, dtype={batch['emb_1'].dtype}")
    if 'emb_2' in batch:
        print(f"  emb_2: shape {batch['emb_2'].shape}, dtype={batch['emb_2'].dtype}")

## Section 2: TorchRec Model Definition

TorchRec provides optimized embedding operations with `EmbeddingBagCollection` for sparse features.

In [ ]:
# Bucketize dense features for embedding lookup
def bucketize_dense_feature(values, bucket_edges):
    """Bucketize dense feature values based on bucket_edges.
    
    Args:
        values: Array of shape (batch_size,)
        bucket_edges: List of bucket edges, e.g., [0.1, 0.2, 0.3, 0.4]
                     Creates buckets: [<0.1, 0.1-0.2, 0.2-0.3, 0.3-0.4, >=0.4]
    
    Returns:
        bucket_indices: Array of shape (batch_size,) with values 0 to len(bucket_edges)
    """
    values = np.asarray(values, dtype=np.float32)
    bucket_indices = np.zeros_like(values, dtype=np.int64)
    
    for i, edge in enumerate(bucket_edges):
        bucket_indices[values >= edge] = i + 1
    
    return bucket_indices

print("Bucketization utility ready")

In [ ]:
# Simple MLP Model (Baseline for comparison)
class SimpleModel(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        
        # Create embedding layers for dense features
        self.dense_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets, config.embedding_dim)
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.DENSE and config.embedding_dim is not None
        })
        
        # Create embedding layers for sparse features
        self.sparse_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets + 1, config.embedding_dim)  # +1 for null bucket 0
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.SPARSE and config.embedding_dim is not None
        })
        
        # Create embedding layers for VarLen sparse features (padding_idx=0)
        self.varlen_sparse_embeddings = nn.ModuleDict({
            name: nn.Embedding(config.num_buckets + 1, config.embedding_dim, padding_idx=0)
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.VAR_LEN_SPARSE and config.embedding_dim is not None
        })
        
        # Store combiner method for each varlen feature
        self.varlen_combiners = {
            name: config.combiner
            for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.VAR_LEN_SPARSE
        }
        
        # Calculate input dimension
        dense_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                           if c.type == FeatureType.DENSE and c.embedding_dim is not None)
        sparse_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                             if c.type == FeatureType.SPARSE and c.embedding_dim is not None)
        varlen_sparse_emb_dim = sum(c.embedding_dim for c in FEATURE_CONFIGS.values() 
                                    if c.type == FeatureType.VAR_LEN_SPARSE and c.embedding_dim is not None)
        existing_emb_dim = sum(c.dim for c in FEATURE_CONFIGS.values() 
                              if c.type == FeatureType.EMBEDDING)
        
        input_dim = dense_emb_dim + sparse_emb_dim + varlen_sparse_emb_dim + existing_emb_dim
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
    
    def pool_varlen_embedding(self, embeddings, indices, combiner='mean'):
        """Pool variable-length sequence embeddings."""
        mask = (indices != 0).unsqueeze(-1).float()
        if combiner == 'sum':
            pooled = (embeddings * mask).sum(dim=1)
        elif combiner == 'mean':
            pooled = (embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        elif combiner == 'max':
            embeddings_masked = embeddings.masked_fill(mask == 0, float('-inf'))
            pooled = embeddings_masked.max(dim=1)[0]
            pooled = pooled.masked_fill(pooled == float('-inf'), 0)
        return pooled
    
    def forward(self, dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings):
        # Embed dense features
        dense_embs = [self.dense_embeddings[name](indices) for name, indices in dense_indices.items()]
        
        # Embed sparse features
        sparse_embs = [self.sparse_embeddings[name](indices) for name, indices in sparse_indices.items()]
        
        # Embed VarLen sparse features with pooling
        varlen_embs = []
        for name, indices in varlen_sparse_indices.items():
            emb = self.varlen_sparse_embeddings[name](indices)
            pooled = self.pool_varlen_embedding(emb, indices, self.varlen_combiners[name])
            varlen_embs.append(pooled)
        
        # Concatenate all embeddings
        x = torch.cat(dense_embs + sparse_embs + varlen_embs + [existing_embeddings], dim=1)
        
        x = F.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))

print("SimpleModel defined")

In [ ]:
# TorchRec DCN Model
class TorchRecDCN(nn.Module):
    """Deep & Cross Network using TorchRec's EmbeddingBagCollection.

    Combines CrossNet (explicit feature crossing) with DNN (deep learning).
    Uses TorchRec's optimized embedding operations.
    """
    def __init__(
        self,
        cross_num: int = 3,
        dnn_hidden: tuple = (256, 128),
        device: torch.device = None
    ):
        super().__init__()
        self.device = device or torch.device("cpu")

        # Prepare feature configs
        dense_configs = {
            name: config for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.DENSE and config.embedding_dim is not None
        }
        sparse_configs = {
            name: config for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.SPARSE and config.embedding_dim is not None
        }
        varlen_configs = {
            name: config for name, config in FEATURE_CONFIGS.items()
            if config.type == FeatureType.VAR_LEN_SPARSE and config.embedding_dim is not None
        }

        # Create EmbeddingBagConfig for dense features (bucketized)
        from torchrec.modules.embedding_configs import PoolingType

        embedding_configs = []

        # Dense features (bucketized -> SparseFeat)
        for name, config in dense_configs.items():
            embedding_configs.append(
                EmbeddingBagConfig(
                    name=name,
                    embedding_dim=config.embedding_dim,
                    num_embeddings=config.num_buckets,
                    feature_names=[name],
                )
            )

        # Sparse features
        for name, config in sparse_configs.items():
            embedding_configs.append(
                EmbeddingBagConfig(
                    name=name,
                    embedding_dim=config.embedding_dim,
                    num_embeddings=config.num_buckets + 1,  # +1 for null bucket 0
                    feature_names=[name],
                )
            )

        # Variable-length sparse features
        for name, config in varlen_configs.items():
            pooling_type = PoolingType.MEAN if config.combiner == 'mean' else (
                PoolingType.SUM if config.combiner == 'sum' else PoolingType.MAX
            )
            embedding_configs.append(
                EmbeddingBagConfig(
                    name=name,
                    embedding_dim=config.embedding_dim,
                    num_embeddings=config.num_buckets + 1,  # +1 for padding idx 0
                    feature_names=[name],
                    pooling=pooling_type,
                )
            )

        # Create EmbeddingBagCollection
        self.embedding_bag_collection = EmbeddingBagCollection(
            tables=embedding_configs,
            device=self.device
        )

        # Calculate total embedding dimension
        dense_emb_dim = sum(c.embedding_dim for c in dense_configs.values())
        sparse_emb_dim = sum(c.embedding_dim for c in sparse_configs.values())
        varlen_sparse_emb_dim = sum(c.embedding_dim for c in varlen_configs.values())
        existing_emb_dim = sum(c.dim for c in FEATURE_CONFIGS.values()
                              if c.type == FeatureType.EMBEDDING)

        total_emb_dim = dense_emb_dim + sparse_emb_dim + varlen_sparse_emb_dim + existing_emb_dim

        # CrossNet
        self.crossnet = TorchRecCrossNet(
            in_features=total_emb_dim,
            num_layers=cross_num
        )

        # DNN tower
        layers = []
        prev_dim = total_emb_dim
        for hidden_dim in dnn_hidden:
            layers.extend([nn.Linear(prev_dim, hidden_dim), nn.ReLU()])
            prev_dim = hidden_dim
        self.dnn = nn.Sequential(*layers)

        # Final output layer
        self.final = nn.Linear(total_emb_dim + dnn_hidden[-1], 1)

    def forward(self, kjt: KeyedJaggedTensor, dense_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            kjt: KeyedJaggedTensor containing sparse feature indices
            dense_features: Pre-computed dense embeddings (emb_1, emb_2 concatenated)
        """
        # Get pooled embeddings from EmbeddingBagCollection
        pooled_embeddings = self.embedding_bag_collection(kjt)

        # Flatten KeyedTensor to single tensor
        # pooled_embeddings is a KeyedTensor, get values() to get concatenated embeddings
        emb_values = pooled_embeddings.values()  # (batch_size, total_sparse_emb_dim)

        # Concatenate with pre-computed dense embeddings
        x = torch.cat([emb_values, dense_features], dim=1)

        # Cross network
        cross_out = self.crossnet(x)

        # DNN
        dnn_out = self.dnn(x)

        # Combine and predict
        stacked = torch.cat([cross_out, dnn_out], dim=-1)
        return torch.sigmoid(self.final(stacked))

print("TorchRecDCN model defined")

## Section 3: KeyedJaggedTensor Construction

TorchRec uses `KeyedJaggedTensor` for efficient sparse feature representation.

In [ ]:
def batch_to_kjt(batch: dict, device: torch.device) -> KeyedJaggedTensor:
    """
    Convert batch dict to TorchRec KeyedJaggedTensor format.
    
    KeyedJaggedTensor is TorchRec's format for variable-length sparse features.
    Each feature is represented as (values, lengths) where:
    - values: flattened indices across all samples
    - lengths: number of indices per sample
    
    Args:
        batch: Dictionary with feature arrays
        device: Target device for tensors
    
    Returns:
        KeyedJaggedTensor containing all sparse features
    """
    keys = []
    values_list = []
    lengths_list = []
    
    # Process dense features (bucketized) - each has length=1 per sample
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.DENSE and config.embedding_dim is not None:
            if name in batch:
                keys.append(name)
                indices = bucketize_dense_feature(batch[name], config.bucket_edges)
                indices_tensor = torch.from_numpy(indices).long()
                values_list.append(indices_tensor.flatten())
                lengths_list.append(torch.ones(len(indices), dtype=torch.int32))
    
    # Process sparse features - each has length=1 per sample
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.SPARSE:
            if name in batch:
                keys.append(name)
                indices = batch[name]
                if isinstance(indices, np.ndarray):
                    indices_tensor = torch.from_numpy(indices).long()
                else:
                    indices_tensor = torch.tensor(indices, dtype=torch.long)
                values_list.append(indices_tensor.flatten())
                lengths_list.append(torch.ones(len(indices), dtype=torch.int32))
    
    # Process variable-length sparse features
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.VAR_LEN_SPARSE:
            if name in batch:
                keys.append(name)
                padded_indices = batch[name]
                if isinstance(padded_indices, np.ndarray):
                    padded_tensor = torch.from_numpy(padded_indices).long()
                else:
                    padded_tensor = torch.tensor(padded_indices, dtype=torch.long)
                
                # Compute lengths (count non-zero elements per row)
                lengths = (padded_tensor != 0).sum(dim=1).int()
                
                # Flatten and filter out padding (keep only non-zero values)
                # For KeyedJaggedTensor, we need to flatten row by row
                batch_size = padded_tensor.shape[0]
                values = []
                for i in range(batch_size):
                    row = padded_tensor[i]
                    non_zero = row[row != 0]
                    values.append(non_zero)
                
                if len(values) > 0:
                    values_list.append(torch.cat(values))
                else:
                    values_list.append(torch.tensor([], dtype=torch.long))
                lengths_list.append(lengths)
    
    # Construct KeyedJaggedTensor
    if len(values_list) == 0:
        # Empty KJT - create dummy
        kjt = KeyedJaggedTensor(
            keys=[],
            values=torch.tensor([], dtype=torch.long, device=device),
            lengths=torch.tensor([], dtype=torch.int32, device=device)
        )
    else:
        kjt = KeyedJaggedTensor(
            keys=keys,
            values=torch.cat(values_list).to(device),
            lengths=torch.cat(lengths_list).to(device)
        )
    
    return kjt

print("KeyedJaggedTensor conversion function ready")

In [ ]:
# Test KeyedJaggedTensor construction
print("Testing KeyedJaggedTensor construction...")
test_batch = ds.take_batch(batch_size=4)
kjt = batch_to_kjt(test_batch, device)

print(f"KJT keys: {kjt.keys()}")
print(f"KJT values shape: {kjt.values().shape}")
print(f"KJT lengths shape: {kjt.lengths().shape}")

# Test with model
test_model = TorchRecDCN(cross_num=3, dnn_hidden=(256, 128), device=device)
dense_features = torch.cat([
    torch.from_numpy(test_batch['emb_1']).float(),
    torch.from_numpy(test_batch['emb_2']).float()
], dim=1).to(device)

with torch.no_grad():
    output = test_model(kjt, dense_features)
    print(f"\nModel forward pass successful!")
    print(f"Output shape: {output.shape}")
    print(f"Sample output: {output[:2].squeeze()}")

## Section 4: Timing Utilities

Device synchronization utilities for accurate timing measurements.

In [ ]:
def sync_and_time(device):
    """Synchronize device and return current time.
    
    Both CUDA and MPS execute operations asynchronously,
    so we must sync before taking timestamps for accurate measurements.
    """
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        # torch.mps.synchronize() requires PyTorch 2.0+
        try:
            torch.mps.synchronize()
        except AttributeError:
            # Fallback: force sync by moving a small tensor to CPU
            _ = torch.zeros(1, device=device).cpu()
    return time.perf_counter()

print("Timing utility ready with device synchronization support")

## Section 5: Single-Worker Training with Throughput Benchmarking

Benchmark single-worker training to establish baseline throughput.

In [3]:
def prepare_input_simple(batch, device):
    """Prepare input for SimpleModel."""
    # Bucketize dense features
    dense_indices = {}
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.DENSE and config.bucket_edges is not None:
            dense_indices[name] = torch.as_tensor(
                bucketize_dense_feature(batch[name], config.bucket_edges)
            ).long().to(device)

    # Sparse features
    sparse_indices = {}
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.SPARSE:
            sparse_indices[name] = torch.as_tensor(batch[name]).long().to(device)

    # VarLen sparse features
    varlen_sparse_indices = {}
    for name, config in FEATURE_CONFIGS.items():
        if config.type == FeatureType.VAR_LEN_SPARSE:
            varlen_sparse_indices[name] = torch.as_tensor(batch[name]).long().to(device)

    # Existing embeddings
    existing_embeddings = torch.cat([
        torch.as_tensor(batch['emb_1']).float(),
        torch.as_tensor(batch['emb_2']).float()
    ], dim=1).to(device)

    # Labels
    y = torch.as_tensor(batch['label']).float().to(device)

    return dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings, y

def prepare_input_torchrec(batch, device):
    """Prepare input for TorchRecDCN."""
    kjt = batch_to_kjt(batch, device)
    dense_features = torch.cat([
        torch.as_tensor(batch['emb_1']).float(),
        torch.as_tensor(batch['emb_2']).float()
    ], dim=1).to(device)
    y = torch.as_tensor(batch['label']).float().to(device)
    return kjt, dense_features, y

print("Input preparation functions ready")

Input preparation functions ready


In [ ]:
def train_with_timing(model, dataset, device, num_batches=200, warmup_batches=10,
                     batch_size=32, model_type='simple'):
    """
    Train model and collect timing metrics.

    Args:
        model: Model to train
        dataset: Ray Dataset
        device: Target device
        num_batches: Number of batches to train on
        warmup_batches: Number of warmup batches to skip
        batch_size: Batch size
        model_type: 'simple' or 'torchrec'

    Returns:
        dict with timing breakdown and throughput metrics
    """
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    data_times, forward_times, backward_times = [], [], []
    total_samples = 0

    # Convert iterable to iterator
    batch_iter = iter(dataset.iter_torch_batches(batch_size=batch_size, prefetch_batches=2))

    for batch_idx in range(num_batches + warmup_batches):
        # Data loading phase
        t0 = sync_and_time(device)
        batch = next(batch_iter)

        if model_type == 'simple':
            dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings, y = prepare_input_simple(batch, device)
        else:  # torchrec
            kjt, dense_features, y = prepare_input_torchrec(batch, device)
        t1 = sync_and_time(device)

        # Forward pass phase
        if model_type == 'simple':
            pred = model(dense_indices, sparse_indices, varlen_sparse_indices, existing_embeddings)
        else:  # torchrec
            pred = model(kjt, dense_features)
        loss = criterion(pred.squeeze(), y)
        t2 = sync_and_time(device)

        # Backward pass phase
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t3 = sync_and_time(device)

        # Skip warmup batches from timing
        if batch_idx >= warmup_batches:
            data_times.append(t1 - t0)
            forward_times.append(t2 - t1)
            backward_times.append(t3 - t2)
            total_samples += y.shape[0]

        # Print progress every 50 batches
        if (batch_idx + 1) % 50 == 0:
            print(f"  Processed {batch_idx + 1} batches...")

    return {
        'data_time': sum(data_times),
        'forward_time': sum(forward_times),
        'backward_time': sum(backward_times),
        'total_samples': total_samples,
        'num_batches': num_batches
    }

print("Training function with timing ready")

In [ ]:
# Use a subset of data for faster benchmarking
# We need enough data for num_batches + warmup_batches (200 + 5)
train_ds = ds.limit((200 + 10) * 32)  # ~210 batches to cover warmup + training

print("=" * 70)
print("Training Simple MLP Model (Single Worker)...")
print("=" * 70)

simple_model = SimpleModel(hidden_dim=128).to(device)
simple_results = train_with_timing(
    simple_model, train_ds, device,
    num_batches=200, warmup_batches=5, batch_size=32, model_type='simple'
)

print(f"\nSimple MLP training completed!")
print(f"  Samples processed: {simple_results['total_samples']}")
print(f"  Data loading time: {simple_results['data_time']:.2f}s")
print(f"  Forward pass time: {simple_results['forward_time']:.2f}s")
print(f"  Backward pass time: {simple_results['backward_time']:.2f}s")

In [ ]:
# Train TorchRec DCN Model (Single Worker)
print("\n" + "=" * 70)
print("Training TorchRec DCN Model (Single Worker)...")
print("=" * 70)

torchrec_model = TorchRecDCN(cross_num=3, dnn_hidden=(256, 128), device=device)
torchrec_results = train_with_timing(
    torchrec_model, train_ds, device,
    num_batches=200, warmup_batches=5, batch_size=32, model_type='torchrec'
)

print(f"\nTorchRec DCN training completed!")
print(f"  Samples processed: {torchrec_results['total_samples']}")
print(f"  Data loading time: {torchrec_results['data_time']:.2f}s")
print(f"  Forward pass time: {torchrec_results['forward_time']:.2f}s")
print(f"  Backward pass time: {torchrec_results['backward_time']:.2f}s")

In [ ]:
# Compute metrics
def compute_metrics(results):
    """Compute derived metrics from timing results."""
    total_time = results['data_time'] + results['forward_time'] + results['backward_time']
    return {
        'total_time': total_time,
        'throughput': results['total_samples'] / total_time if total_time > 0 else 0,
        'data_wait_ratio': results['data_time'] / total_time if total_time > 0 else 0,
        'forward_ratio': results['forward_time'] / total_time if total_time > 0 else 0,
        'backward_ratio': results['backward_time'] / total_time if total_time > 0 else 0,
        **results
    }

simple_metrics = compute_metrics(simple_results)
torchrec_metrics = compute_metrics(torchrec_results)

print("Metrics computed!")

In [ ]:
def print_comparison(simple_metrics, torchrec_metrics):
    """Print side-by-side comparison table."""
    print("\n" + "╔" + "═" * 75 + "╗")
    print("║" + " " * 20 + "Single-Worker Training Benchmark" + " " * 28 + "║")
    print("╠" + "═" * 75 + "╣")
    
    # Header
    print("║ {:<25} │ {:<20} │ {:<20} ║".format("Metric", "Simple MLP", "TorchRec DCN"))
    print("╠" + "═" * 25 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╣")
    
    # Total samples
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "Total samples",
        f"{simple_metrics['total_samples']:,}",
        f"{torchrec_metrics['total_samples']:,}"
    ))
    
    # Total time
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "Total time",
        f"{simple_metrics['total_time']:.2f}s",
        f"{torchrec_metrics['total_time']:.2f}s"
    ))
    
    print("╠" + "─" * 25 + "╪" + "─" * 20 + "╪" + "─" * 20 + "╣")
    print("║ {:<25} │ {:<20} │ {:<20} ║".format("Time Breakdown:", "", ""))
    
    # Data loading
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "  Data loading",
        f"{simple_metrics['data_time']:.2f}s ({simple_metrics['data_wait_ratio']*100:.1f}%)",
        f"{torchrec_metrics['data_time']:.2f}s ({torchrec_metrics['data_wait_ratio']*100:.1f}%)"
    ))
    
    # Forward pass
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "  Forward pass",
        f"{simple_metrics['forward_time']:.2f}s ({simple_metrics['forward_ratio']*100:.1f}%)",
        f"{torchrec_metrics['forward_time']:.2f}s ({torchrec_metrics['forward_ratio']*100:.1f}%)"
    ))
    
    # Backward pass
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "  Backward pass",
        f"{simple_metrics['backward_time']:.2f}s ({simple_metrics['backward_ratio']*100:.1f}%)",
        f"{torchrec_metrics['backward_time']:.2f}s ({torchrec_metrics['backward_ratio']*100:.1f}%)"
    ))
    
    print("╠" + "═" * 25 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╣")
    
    # Throughput
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "Throughput",
        f"{simple_metrics['throughput']:.1f} samples/s",
        f"{torchrec_metrics['throughput']:.1f} samples/s"
    ))
    
    # Data wait ratio
    simple_bound = "I/O-bound" if simple_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    torchrec_bound = "I/O-bound" if torchrec_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    print("║ {:<25} │ {:<20} │ {:<20} ║".format(
        "Data wait ratio",
        f"{simple_metrics['data_wait_ratio']:.3f} ({simple_bound})",
        f"{torchrec_metrics['data_wait_ratio']:.3f} ({torchrec_bound})"
    ))
    
    print("╚" + "═" * 75 + "╝")
    
    # Interpretation
    print("\nInterpretation:")
    print(f"  - Simple MLP:    {simple_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {simple_bound}")
    print(f"  - TorchRec DCN:  {torchrec_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {torchrec_bound}")

print_comparison(simple_metrics, torchrec_metrics)

## Section 6: Distributed Training with Ray Train

Use Ray Train for multi-worker distributed training with automatic sharding.

In [ ]:
def train_loop_per_worker(config):
    """
    Training loop that runs on each Ray worker.

    This function is executed on each worker process. Ray Train automatically
    shards the dataset and distributes it across workers.
    """
    import torch
    import torch.nn as nn

    # Get dataset shard for this worker
    train_data = get_dataset_shard("train")

    # Create model
    model = TorchRecDCN(
        cross_num=config["cross_num"],
        dnn_hidden=config["dnn_hidden"],
        device=torch.device("cpu")  # Ray Train handles device placement
    )

    # Prepare model for distributed training
    model = train.torch.prepare_model(model)

    # Get device from wrapped model
    device = next(model.parameters()).device

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.BCELoss()

    # Timing instrumentation
    data_times, forward_times, backward_times = [], [], []
    total_samples = 0

    num_batches = config["num_batches"]
    warmup_batches = config["warmup_batches"]
    batch_size = config["batch_size"]

    # Iterate over batches
    # Convert iterable to iterator explicitly
    batch_iter = iter(train_data.iter_torch_batches(batch_size=batch_size, prefetch_batches=2))

    for batch_idx in range(num_batches + warmup_batches):
        # Data loading phase
        t0 = time.perf_counter()
        batch = next(batch_iter)
        kjt, dense_features, y = prepare_input_torchrec(batch, device)
        t1 = time.perf_counter()

        # Forward pass phase
        pred = model(kjt, dense_features)
        loss = criterion(pred.squeeze(), y)
        t2 = time.perf_counter()

        # Backward pass phase
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t3 = time.perf_counter()

        # Skip warmup batches from timing
        if batch_idx >= warmup_batches:
            data_times.append(t1 - t0)
            forward_times.append(t2 - t1)
            backward_times.append(t3 - t2)
            total_samples += y.shape[0]

        # Report metrics periodically
        if (batch_idx + 1) % 50 == 0:
            current_loss = loss.item()
            total_time = sum(data_times) + sum(forward_times) + sum(backward_times)
            throughput = total_samples / total_time if total_time > 0 else 0
            data_ratio = sum(data_times) / total_time if total_time > 0 else 0

            train.report({
                "loss": current_loss,
                "throughput": throughput,
                "data_wait_ratio": data_ratio,
                "batch_idx": batch_idx + 1,
            })

    # Final metrics
    total_time = sum(data_times) + sum(forward_times) + sum(backward_times)
    final_throughput = total_samples / total_time if total_time > 0 else 0
    final_data_ratio = sum(data_times) / total_time if total_time > 0 else 0

    train.report({
        "final_loss": loss.item(),
        "final_throughput": final_throughput,
        "final_data_wait_ratio": final_data_ratio,
        "total_samples": total_samples,
        "data_time": sum(data_times),
        "forward_time": sum(forward_times),
        "backward_time": sum(backward_times),
    })

print("Ray Train loop function ready")

In [ ]:
# Configure and run distributed training
print("=" * 70)
print("Training TorchRec DCN Model (Distributed - 2 Workers)...")
print("=" * 70)

# Prepare dataset for Ray Train
# Need enough data for 2 workers: (num_batches + warmup) * batch_size * num_workers
train_dataset = ds.limit((200 + 10) * 32 * 2) 

trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "cross_num": 3,
        "dnn_hidden": (256, 128),
        "lr": 1e-3,
        "batch_size": 32,
        "num_batches": 200,
        "warmup_batches": 5,
    },
    scaling_config=ScalingConfig(
        num_workers=2,        # Number of parallel workers
        use_gpu=False,        # Set to True if GPUs available
    ),
    datasets={"train": train_dataset},
)

# Run training
result = trainer.fit()

print(f"\nDistributed training completed!")
print(f"Final metrics: {result.metrics}")

In [ ]:
# Manually populate metrics because Ray failed to collect them due to connection issues
# Values are extracted from the worker logs printed in the output above

# Worker 0 reported: throughput=1155.2, data_time=1.81, forward=0.85, backward=2.88
# Worker 1 reported: throughput=1155.8, data_time=1.71, forward=0.87, backward=2.95

# We aggregate these for the total system metrics
distributed_metrics = {
    'total_samples': 6400 * 2,    # 6400 samples per worker * 2 workers
    'data_time': 1.76,            # Average of two workers
    'forward_time': 0.86,         # Average of two workers
    'backward_time': 2.92,        # Average of two workers
    'throughput': 1155.2 + 1155.8,# Sum of both workers
    'data_wait_ratio': 0.318,     # Average
}

# Compute total loop time (average per worker)
distributed_metrics['total_time'] = (
    distributed_metrics['data_time'] +
    distributed_metrics['forward_time'] +
    distributed_metrics['backward_time']
)

print("Distributed metrics manually loaded from logs.")
print(f"Metrics: {distributed_metrics}")

In [ ]:
# Extract distributed training metrics
if result.metrics is not None:
    distributed_metrics = {
        'total_samples': result.metrics.get('total_samples', 0),
        'data_time': result.metrics.get('data_time', 0),
        'forward_time': result.metrics.get('forward_time', 0),
        'backward_time': result.metrics.get('backward_time', 0),
        'throughput': result.metrics.get('final_throughput', 0),
        'data_wait_ratio': result.metrics.get('final_data_wait_ratio', 0),
    }

    # Compute total time
    distributed_metrics['total_time'] = (
        distributed_metrics['data_time'] +
        distributed_metrics['forward_time'] +
        distributed_metrics['backward_time']
    )
    print("Distributed training metrics extracted from result object")
else:
    print("result.metrics is None (Ray connection issue).")
    if 'distributed_metrics' in locals():
        print("Using manually loaded distributed metrics from previous cell.")
        print(f"Metrics: {distributed_metrics}")
    else:
        print("Error: distributed_metrics not defined. Please run the manual loading cell first.")

In [ ]:
# Three-way comparison: Simple MLP vs TorchRec DCN (single) vs TorchRec DCN (distributed)
def print_three_way_comparison(simple_metrics, torchrec_metrics, distributed_metrics):
    """Print three-way comparison table."""
    print("\n" + "╔" + "═" * 95 + "╗")
    print("║" + " " * 25 + "Training Benchmark: Single vs Distributed" + " " * 27 + "║")
    print("╠" + "═" * 95 + "╣")
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Metric", "Simple MLP", "TorchRec DCN (1W)", "TorchRec DCN (2W)"))
    print("╠" + "═" * 25 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╣")
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Total samples",
        f"{simple_metrics['total_samples']:,}",
        f"{torchrec_metrics['total_samples']:,}",
        f"{distributed_metrics.get('total_samples', 0):,}"
    ))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Total time",
        f"{simple_metrics['total_time']:.2f}s",
        f"{torchrec_metrics['total_time']:.2f}s",
        f"{distributed_metrics.get('total_time', 0):.2f}s"
    ))
    
    print("╠" + "─" * 25 + "╪" + "─" * 20 + "╪" + "─" * 20 + "╪" + "─" * 20 + "╣")
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format("Time Breakdown:", "", "", ""))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "  Data loading",
        f"{simple_metrics['data_time']:.2f}s ({simple_metrics['data_wait_ratio']*100:.1f}%)",
        f"{torchrec_metrics['data_time']:.2f}s ({torchrec_metrics['data_wait_ratio']*100:.1f}%)",
        f"{distributed_metrics.get('data_time', 0):.2f}s ({distributed_metrics.get('data_wait_ratio', 0)*100:.1f}%)"
    ))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "  Forward pass",
        f"{simple_metrics['forward_time']:.2f}s ({simple_metrics['forward_ratio']*100:.1f}%)",
        f"{torchrec_metrics['forward_time']:.2f}s ({torchrec_metrics['forward_ratio']*100:.1f}%)",
        f"{distributed_metrics.get('forward_time', 0):.2f}s"
    ))
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "  Backward pass",
        f"{simple_metrics['backward_time']:.2f}s ({simple_metrics['backward_ratio']*100:.1f}%)",
        f"{torchrec_metrics['backward_time']:.2f}s ({torchrec_metrics['backward_ratio']*100:.1f}%)",
        f"{distributed_metrics.get('backward_time', 0):.2f}s"
    ))
    
    print("╠" + "═" * 25 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╪" + "═" * 20 + "╣")
    
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Throughput",
        f"{simple_metrics['throughput']:.1f} samp/s",
        f"{torchrec_metrics['throughput']:.1f} samp/s",
        f"{distributed_metrics.get('throughput', 0):.1f} samp/s"
    ))
    
    simple_bound = "I/O-bound" if simple_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    torchrec_bound = "I/O-bound" if torchrec_metrics['data_wait_ratio'] > 0.5 else "Compute-bound"
    distributed_bound = "I/O-bound" if distributed_metrics.get('data_wait_ratio', 0) > 0.5 else "Compute-bound"
    print("║ {:<25} │ {:<20} │ {:<20} │ {:<20} ║".format(
        "Data wait ratio",
        f"{simple_metrics['data_wait_ratio']:.3f} ({simple_bound})",
        f"{torchrec_metrics['data_wait_ratio']:.3f} ({torchrec_bound})",
        f"{distributed_metrics.get('data_wait_ratio', 0):.3f} ({distributed_bound})"
    ))
    
    print("╚" + "═" * 95 + "╝")
    
    # Interpretation
    print("\nInterpretation:")
    print(f"  - Simple MLP:           {simple_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {simple_bound}")
    print(f"  - TorchRec DCN (1W):    {torchrec_metrics['data_wait_ratio']*100:.1f}% time waiting for data → {torchrec_bound}")
    print(f"  - TorchRec DCN (2W):    {distributed_metrics.get('data_wait_ratio', 0)*100:.1f}% time waiting for data → {distributed_bound}")
    print("\nKey Insight:")
    print("  Distributed training with Ray Train enables parallel data loading and")
    print("  computation across multiple workers, potentially improving throughput.")

print_three_way_comparison(simple_metrics, torchrec_metrics, distributed_metrics)

## Section 7: Cleanup

Shutdown Ray and clean up resources.

In [ ]:
# Shutdown Ray
ray.shutdown()

print("Ray shutdown complete")